In [2]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import List
from collections import defaultdict

In [3]:
# load environment variables from .env file
load_dotenv()

True

In [4]:
# Define the persistent directory for Chroma database
persistent_directory = "db/chroma_db"

# Load embeddings and vector store
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Initialize Chroma vector store with the specified persistent directory and embedding model
db = Chroma(
    persist_directory=persistent_directory,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space": "cosine"},
)

# load the Google Gemini model for generating responses
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5815.85it/s]


In [5]:
# Pydantic model for structured output
class QueryVariations(BaseModel):
    queries: List[str]

In [6]:
# ──────────────────────────────────────────────────────────────────
# MAIN EXECUTION
# ──────────────────────────────────────────────────────────────────

# Original query
original_query = "How does Tesla make money?"
print(f"Original Query: {original_query}\n")

Original Query: How does Tesla make money?



In [7]:
# ──────────────────────────────────────────────────────────────────
# Step 1: Generate Multiple Query Variations
# ──────────────────────────────────────────────────────────────────

llm_with_tools = llm.with_structured_output(QueryVariations)

prompt = f"""Generate 3 different variations of this query that would help retrieve relevant documents:

Original query: {original_query}

Return 3 alternative queries that rephrase or approach the same question from different angles."""

response = llm_with_tools.invoke(prompt)

query_variations = response.queries

print("Generated Query Variations:")
for i, variation in enumerate(query_variations, start=1):
    print(f"{i}. {variation}")

print("\n" + "=" * 60)

Generated Query Variations:
1. What are Tesla's primary revenue sources?
2. Describe Tesla's business model and how it generates profit.
3. What are the different ways Tesla earns income?



In [8]:
print(query_variations)
print(len(query_variations))

["What are Tesla's primary revenue sources?", "Describe Tesla's business model and how it generates profit.", 'What are the different ways Tesla earns income?']
3


In [9]:
# ──────────────────────────────────────────────────────────────────
# Step 2: Search with Each Query Variation & Store Results
# ──────────────────────────────────────────────────────────────────

retriever = db.as_retriever(search_kwargs={"k": 5})  # Get more docs for better RRF

all_retrieval_results = []  # Store all results for RRF

for i, query in enumerate(query_variations, 1):
    print(f"\n=== RESULTS FOR QUERY {i}: {query} ===")

    docs = retriever.invoke(query)
    all_retrieval_results.append(docs)  # Store for RRF calculation

    print(f"Retrieved {len(docs)} documents:\n")

    for j, doc in enumerate(docs, 1):
        print(f"Document {j}:")
        print(f"{doc.page_content[:150]}...\n")

    print("-" * 50)

print("\n" + "=" * 60)
print("Multi-Query Retrieval Complete!")
print("Notice how different query variations retrieved different documents.")


=== RESULTS FOR QUERY 1: What are Tesla's primary revenue sources? ===
Retrieved 5 documents:

Document 1:
Finances
For the fiscal (and calendar) year 2021, Tesla
reported a net income of $5.52 billion.[562] The Tesla financial performance

annual revenue w...

Document 2:
60. Root, Al (December 7, 2020). "Tesla Becomes Only the Sixth Company to Top $600 Billion in Value" (https://
www.barrons.com/articles/tesla-becomes-...

Document 3:
Production  1,773,443 vehicles (2024)
A lawsuit settlement agreed to by Eberhard and Tesla in September 2009 output  31.4 GWh battery energy
allows al...

Document 4:
568. O'Kane, Sean (July 26, 2021). "Tesla finally made a profit without the help of emission credits" (https://www.t
heverge.com/2021/7/26/22594778/te...

Document 5:
Tesla was incorporated in July 2003 by Martin Eberhard and Marc
Tarpenning as Tesla Motors. Its name is a tribute to inventor and
electrical engineer ...

--------------------------------------------------

=== RESULTS FOR Q

In [10]:
# melihat jumlah dokumen yang diambil dari setiap query disimpan dalam all_retrieval_results
for i, docs in enumerate(all_retrieval_results, start=1):
    print(f"Query {i} retrieved {len(docs)} documents.")

Query 1 retrieved 5 documents.
Query 2 retrieved 5 documents.
Query 3 retrieved 5 documents.


In [11]:
# ──────────────────────────────────────────────────────────────────
# Step 3: Apply Reciprocal Rank Fusion (RRF)
# ──────────────────────────────────────────────────────────────────

def reciprocal_rank_fusion(chunk_lists, k=60, verbose=True):

    if verbose:
        print("\n" + "=" * 60)
        print("APPLYING RECIPROCAL RANK FUSION")
        print("=" * 60)
        print(f"\nUsing k={k}")
        print("Calculating RRF scores...\n")

    # Data structures for RRF calculation
    rrf_scores = defaultdict(float)  # Will store: {chunk_content: rrf_score}
    all_unique_chunks = {}  # Will store: {chunk_content: actual_chunk_object}

    # For verbose output - track chunk IDs
    chunk_id_map = {}
    chunk_counter = 1

    # Go through each retrieval result
    for query_idx, chunks in enumerate(chunk_lists, 1):
        if verbose:
            print(f"Processing Query {query_idx} results:")

        # Go through each chunk in this query's results
        for position, chunk in enumerate(chunks, 1):  # position is 1-indexed
            # Use chunk content as unique identifier
            chunk_content = chunk.page_content

            # Assign a simple ID if we haven't seen this chunk before
            if chunk_content not in chunk_id_map:
                chunk_id_map[chunk_content] = f"Chunk_{chunk_counter}"
                chunk_counter += 1

            chunk_id = chunk_id_map[chunk_content]

            # Store the chunk object (in case we haven't seen it before)
            all_unique_chunks[chunk_content] = chunk

            # Calculate position score: 1/(k + position)
            position_score = 1 / (k + position)

            # Add to RRF score
            rrf_scores[chunk_content] += position_score

            if verbose:
                print(
                    f"  Position {position}: {chunk_id} +{position_score:.4f} (running total: {rrf_scores[chunk_content]:.4f})"
                )
                print(f"    Preview: {chunk_content[:80]}...")

        if verbose:
            print()

    # Sort chunks by RRF score (highest first)
    sorted_chunks = sorted(
        [
            (all_unique_chunks[chunk_content], score)
            for chunk_content, score in rrf_scores.items()
        ],
        key=lambda x: x[1],  # Sort by RRF score
        reverse=True,  # Highest scores first
    )

    if verbose:
        print(
            f"✅ RRF Complete! Processed {len(sorted_chunks)} unique chunks from {len(chunk_lists)} queries."
        )

    return sorted_chunks


# Apply RRF to our retrieval results
fused_results = reciprocal_rank_fusion(all_retrieval_results, k=60, verbose=True)


APPLYING RECIPROCAL RANK FUSION

Using k=60
Calculating RRF scores...

Processing Query 1 results:
  Position 1: Chunk_1 +0.0164 (running total: 0.0164)
    Preview: Finances
For the fiscal (and calendar) year 2021, Tesla
reported a net income of...
  Position 2: Chunk_2 +0.0161 (running total: 0.0161)
    Preview: 60. Root, Al (December 7, 2020). "Tesla Becomes Only the Sixth Company to Top $6...
  Position 3: Chunk_3 +0.0159 (running total: 0.0159)
    Preview: Production  1,773,443 vehicles (2024)
A lawsuit settlement agreed to by Eberhard...
  Position 4: Chunk_4 +0.0156 (running total: 0.0156)
    Preview: 568. O'Kane, Sean (July 26, 2021). "Tesla finally made a profit without the help...
  Position 5: Chunk_5 +0.0154 (running total: 0.0154)
    Preview: Tesla was incorporated in July 2003 by Martin Eberhard and Marc
Tarpenning as Te...

Processing Query 2 results:
  Position 1: Chunk_5 +0.0164 (running total: 0.0318)
    Preview: Tesla was incorporated in July 2003 by Martin Ebe

In [12]:
# ──────────────────────────────────────────────────────────────────
# Step 4: Display Final Fused Results
# ──────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("FINAL RRF RANKING")
print("=" * 60)

print(f"\nTop {min(10, len(fused_results))} documents after RRF fusion:\n")

for rank, (doc, rrf_score) in enumerate(fused_results[:10], 1):
    print(f"🏆 RANK {rank} (RRF Score: {rrf_score:.4f})")
    print(f"{doc.page_content[:200]}...")
    print("-" * 50)

print(
    f"\n✅ RRF Complete! Fused {len(fused_results)} unique documents from {len(query_variations)} query variations."
)
print("\n💡 Key benefits:")
print("   • Documents appearing in multiple queries get boosted scores")
print("   • Higher positions contribute more to the final score")
print("   • Balanced fusion using k=60 for gentle position penalties")

# ──────────────────────────────────────────────────────────────────
# Optional: Quick Usage Examples
# ──────────────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("USAGE EXAMPLES")
print("=" * 60)



FINAL RRF RANKING

Top 10 documents after RRF fusion:

🏆 RANK 1 (RRF Score: 0.0484)
Finances
For the fiscal (and calendar) year 2021, Tesla
reported a net income of $5.52 billion.[562] The Tesla financial performance

annual revenue was $53.8  billion, an increase
of 71% over the pre...
--------------------------------------------------
🏆 RANK 2 (RRF Score: 0.0318)
Tesla was incorporated in July 2003 by Martin Eberhard and Marc
Tarpenning as Tesla Motors. Its name is a tribute to inventor and
electrical engineer Nikola Tesla. In February 2004, Elon Musk led
Tesl...
--------------------------------------------------
🏆 RANK 3 (RRF Score: 0.0315)
Production  1,773,443 vehicles (2024)
A lawsuit settlement agreed to by Eberhard and Tesla in September 2009 output  31.4 GWh battery energy
allows all five – Eberhard, Tarpenning, Wright, Musk, and S...
--------------------------------------------------
🏆 RANK 4 (RRF Score: 0.0161)
60. Root, Al (December 7, 2020). "Tesla Becomes Only the Sixth 